# ⚡ Quick Run Mode: Fast vs Full

This notebook supports a global run mode (EQUIPAY_MODE) for FAST/FULL runs.


In [ ]:
# GLOBAL RUN MODE (inserted)
import os
EQUIPAY_MODE = os.environ.get('EQUIPAY_MODE', 'FAST')  # FAST | FULL
if EQUIPAY_MODE == 'FAST':
    N_SAMPLES = 1000
    N_BOOTSTRAP = 100
    MAX_ITER = 200
    N_ESTIMATORS = 50
    PLOT_INLINE = False
else:
    N_SAMPLES = None
    N_BOOTSTRAP = 1000
    MAX_ITER = 1000
    N_ESTIMATORS = 200
    PLOT_INLINE = True
print(f"EQUIPAY_MODE={EQUIPAY_MODE}; N_SAMPLES={N_SAMPLES}; N_BOOTSTRAP={N_BOOTSTRAP}")


In [ ]:
# RUN-MODE UTILITIES (safe)
import os
EQUIPAY_MODE = globals().get('EQUIPAY_MODE', os.environ.get('EQUIPAY_MODE','FAST'))

# Better defaults: 10K for FAST mode, 100K for FULL mode  
if EQUIPAY_MODE == 'FAST':
    N_SAMPLES = globals().get('N_SAMPLES', 10000)
else:
    N_SAMPLES = globals().get('N_SAMPLES', 100000)

N_BOOTSTRAP = globals().get('N_BOOTSTRAP', 100)
MAX_ITER = globals().get('MAX_ITER', 200)
N_ESTIMATORS = globals().get('N_ESTIMATORS', 50)

def enforce_fast_sample(df, n=None, seed=42):
    if EQUIPAY_MODE == 'FAST' and n is not None:
        return df.sample(n=min(len(df), n), random_state=seed)
    return df

# Conservative default: in FAST mode we skip small groups to avoid heavy per-group loops.
# In FULL mode we allow small groups to be processed for comprehensive analysis.
MIN_GROUP_N = 1000 if EQUIPAY_MODE == 'FAST' else 30
print(f"EQUIPAY_MODE={EQUIPAY_MODE}; MIN_GROUP_N={MIN_GROUP_N}")

from src.notebook_utils import ensure_store_and_sample, get_sample_from_store, safe_weight_col
store, df_sample = ensure_store_and_sample(sample_limit=N_SAMPLES)
weight_col = safe_weight_col(df_sample)

# Data loading feedback
print(f"✓ Loaded {len(df_sample):,} records")
print(f"✓ Dataset has {len(df_sample.columns)} columns")
if 'SURVYEAR' in df_sample.columns:
    year_range = f"{df_sample['SURVYEAR'].min():.0f}-{df_sample['SURVYEAR'].max():.0f}"
    print(f"✓ Year range: {year_range}")

In [ ]:
# ============================================================================
# SETUP AND IMPORTS
# ============================================================================

import warnings
warnings.filterwarnings('ignore')

import sys
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm

from data_store import EquiPayDataStore, Agg, Func
from src.gap_analysis.policy import (
    DifferenceInDifferences,
    EventStudy,
    TripleDifference,
    CANADIAN_POLICY_EVENTS,
    analyze_policy_impact
)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)

print("✅ Imports successful (6-Layer Data Store Architecture)")
print(f"\n📋 Available policy events: {list(CANADIAN_POLICY_EVENTS.keys())}")

In [ ]:
# ============================================================================
# SETUP: Import Libraries and Configure Environment
# ============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import warnings
warnings.filterwarnings('ignore')

# Statistical models
import statsmodels.api as sm
from statsmodels.regression.linear_model import WLS
from scipy import stats

# Project imports
from src.constants import COLS, normalize_column_names, PROV_LABELS

# Policy analysis modules
from src.gap_analysis.policy import (
    DifferenceInDifferences,
    EventStudy,
    TripleDifference,
    CANADIAN_POLICY_EVENTS,
    analyze_policy_impact,
)
from src.gap_analysis.core import weighted_mean

# Publication-quality figures
plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'font.size': 11,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'figure.facecolor': 'white',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

print("✓ Libraries loaded successfully")
print("✓ Policy analysis modules loaded")
print(f"\nAvailable policy events: {list(CANADIAN_POLICY_EVENTS.keys())}")

In [ ]:
# ============================================================================
# DATA LOADING VIA EquiPayDataStore (6-Layer Architecture)
# ============================================================================

# Initialize data store with new architecture
store = EquiPayDataStore(
    parquet_path=str(PROJECT_ROOT / "data" / "parquet"),
    memory_limit_mb=1000,
    enable_cache=True
)

# Load data with valid wages using new API
try:
    df = store.query().where('HRLYEARN > 0').to_pandas()
except Exception:
    df = get_sample_from_store(query='HRLYEARN > 0', limit=N_SAMPLES)

df = normalize_column_names(df)

# Create analysis variables
df['IS_FEMALE'] = (df['GENDER'] == 2).astype(int)
df['LOG_WAGE'] = np.log(df['REAL_HRLYEARN'].clip(lower=1))
df['PROV_LABEL'] = df['PROV'].map(PROV_LABELS)

# Get summary stats
total_records = store.count()
years = store.years()

print(f"✓ Loaded {len(df):,} records (6-Layer Architecture)")
print(f"✓ Years: {min(years)} - {max(years)}")

## 1. Descriptive Analysis: Gender Gap Trends by Province

First, visualize how the gender gap evolved differently across provinces.

In [ ]:
# ============================================================================
# GENDER GAP TRENDS BY PROVINCE
# ============================================================================

print("=" * 70)
print("GENDER GAP TRENDS BY PROVINCE")
print("=" * 70)

# Calculate annual gender gap by province
gap_trends = []

for year in sorted(df['SURVYEAR'].unique()):
    for prov in [35, 24, 59, 48]:  # ON, QC, BC, AB
        subset = df[(df['SURVYEAR'] == year) & (df['PROV'] == prov)]
        if len(subset) < 500:
            continue
        
        male = subset[subset['GENDER'] == 1]
        female = subset[subset['GENDER'] == 2]
        
        male_wage = weighted_mean(male['REAL_HRLYEARN'].values, male['FINALWT'].values)
        female_wage = weighted_mean(female['REAL_HRLYEARN'].values, female['FINALWT'].values)
        
        gap = (female_wage - male_wage) / male_wage * 100
        
        gap_trends.append({
            'Year': year,
            'Province': PROV_LABELS.get(prov, str(prov)),
            'PROV': prov,
            'Gap (%)': gap
        })

gap_trends_df = pd.DataFrame(gap_trends)

# Visualization
fig, ax = plt.subplots(figsize=(12, 6))

for prov in gap_trends_df['Province'].unique():
    prov_data = gap_trends_df[gap_trends_df['Province'] == prov]
    ax.plot(prov_data['Year'], prov_data['Gap (%)'], 'o-', label=prov, linewidth=2)

# Add policy event markers
ax.axvline(x=2020, color='red', linestyle='--', alpha=0.5, label='COVID-19')
ax.axvline(x=2021, color='green', linestyle='--', alpha=0.5, label='Federal Pay Equity Act')

ax.axhline(y=0, color='black', linewidth=0.8)
ax.set_xlabel('Year')
ax.set_ylabel('Gender Gap (%)')
ax.set_title('Gender Wage Gap by Province Over Time', fontweight='bold')
ax.legend(loc='best')

plt.tight_layout()
plt.savefig('../reports/figures/gap_trends_by_province.png', dpi=300, bbox_inches='tight')
plt.show()

## 2. Difference-in-Differences: Federal Pay Equity Act (2021)

Evaluate the effect of the Federal Pay Equity Act by comparing federally-regulated vs provincially-regulated workers.

In [ ]:
# ============================================================================
# DIFFERENCE-IN-DIFFERENCES: Federal Pay Equity Act 2021
# ============================================================================

print("=" * 70)
print("DIFFERENCE-IN-DIFFERENCES: Federal Pay Equity Act (2021)")
print("=" * 70)
print("""
Design:
- Treatment: Post-August 2021
- Treated group: Public sector (proxy for federal regulation)
- Control group: Private sector (provincial regulation)
- Outcome: Gender gap in log wages
""")

# Define treatment groups
# COWMAIN: 1 = Public, 2 = Private
df['IS_PUBLIC'] = (df['COWMAIN'] == 1).astype(int)
df['POST_2021'] = (df['SURVYEAR'] >= 2021).astype(int)
df['TREATMENT'] = df['IS_PUBLIC'] * df['POST_2021']

# Focus on recent years for cleaner identification
df_did = df[df['SURVYEAR'].isin([2018, 2019, 2020, 2021, 2022, 2023, 2024])].copy()

# Calculate 2x2 table
print("\n--- 2x2 DiD Table (Mean Gender Gap %) ---")
print()

gaps_2x2 = {}
for sector in ['Public', 'Private']:
    is_public = 1 if sector == 'Public' else 0
    for period in ['Pre-2021', 'Post-2021']:
        is_post = 1 if period == 'Post-2021' else 0
        
        subset = df_did[(df_did['IS_PUBLIC'] == is_public) & 
                        (df_did['POST_2021'] == is_post)]
        
        male = subset[subset['GENDER'] == 1]
        female = subset[subset['GENDER'] == 2]
        
        male_wage = weighted_mean(male['LOG_WAGE'].values, male['FINALWT'].values)
        female_wage = weighted_mean(female['LOG_WAGE'].values, female['FINALWT'].values)
        
        gap = (np.exp(female_wage - male_wage) - 1) * 100
        gaps_2x2[(sector, period)] = gap

print(f"{'Sector':<15} {'Pre-2021':>12} {'Post-2021':>12} {'Diff':>12}")
print("-" * 55)
for sector in ['Public', 'Private']:
    pre = gaps_2x2[(sector, 'Pre-2021')]
    post = gaps_2x2[(sector, 'Post-2021')]
    diff = post - pre
    print(f"{sector:<15} {pre:>12.1f}% {post:>12.1f}% {diff:>+12.1f}%")

# DiD estimate
did_estimate = ((gaps_2x2[('Public', 'Post-2021')] - gaps_2x2[('Public', 'Pre-2021')]) -
                (gaps_2x2[('Private', 'Post-2021')] - gaps_2x2[('Private', 'Pre-2021')]))

print("-" * 55)
print(f"\n{'DiD Estimate:':<30} {did_estimate:>+.2f} percentage points")
print(f"\nInterpretation: The Federal Pay Equity Act is associated with a")
print(f"{abs(did_estimate):.1f} pp {'reduction' if did_estimate > 0 else 'widening'} in the public sector gender gap")
print(f"relative to the private sector.")

In [ ]:
# ============================================================================
# REGRESSION-BASED DiD WITH CONTROLS
# ============================================================================

print("\n" + "=" * 70)
print("REGRESSION-BASED DiD WITH CONTROLS")
print("=" * 70)
print("\nModel: ln(Wage) = β₀ + β₁Female + β₂Public + β₃Post + β₄(Female×Public×Post) + γX + ε")

# Prepare data
df_reg = df_did[['LOG_WAGE', 'IS_FEMALE', 'IS_PUBLIC', 'POST_2021',
                  'EDUC', 'AGE_12', 'TENURE', 'FINALWT']].dropna()

# Create triple interaction
df_reg['FEMALE_X_PUBLIC'] = df_reg['IS_FEMALE'] * df_reg['IS_PUBLIC']
df_reg['FEMALE_X_POST'] = df_reg['IS_FEMALE'] * df_reg['POST_2021']
df_reg['PUBLIC_X_POST'] = df_reg['IS_PUBLIC'] * df_reg['POST_2021']
df_reg['DID_TERM'] = df_reg['IS_FEMALE'] * df_reg['IS_PUBLIC'] * df_reg['POST_2021']

# Features
features = ['IS_FEMALE', 'IS_PUBLIC', 'POST_2021', 
            'FEMALE_X_PUBLIC', 'FEMALE_X_POST', 'PUBLIC_X_POST',
            'DID_TERM', 'EDUC', 'AGE_12', 'TENURE']

X = df_reg[features]
X = sm.add_constant(X)
y = df_reg['LOG_WAGE']
weights = df_reg['FINALWT']

model = WLS(y, X, weights=weights).fit(cov_type='HC3')

# Display key coefficients
print("\n" + "=" * 60)
print(f"{'Variable':<25} {'Coef':>10} {'SE':>10} {'p-value':>10}")
print("=" * 60)

key_vars = ['IS_FEMALE', 'DID_TERM']
for var in key_vars:
    coef = model.params[var]
    se = model.bse[var]
    p = model.pvalues[var]
    pct = (np.exp(coef) - 1) * 100
    sig = '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.1 else ''
    print(f"{var:<25} {coef:>10.4f} {se:>10.4f} {p:>10.4f} {sig}")
    print(f"  → {pct:+.1f}% wage effect")

print("\n" + "=" * 60)
print(f"R²: {model.rsquared:.4f}")
print(f"N: {len(df_reg):,}")

# Interpret DID coefficient
did_coef = model.params['DID_TERM']
did_pct = (np.exp(did_coef) - 1) * 100
did_p = model.pvalues['DID_TERM']

print(f"\n" + "=" * 60)
print("POLICY EFFECT INTERPRETATION")
print("=" * 60)
print(f"\nDiD estimate (Female×Public×Post): {did_pct:+.1f}%")
print(f"p-value: {did_p:.4f}")

if did_p < 0.05:
    if did_coef > 0:
        print(f"\n✓ POLICY EFFECTIVE: Federal Pay Equity Act significantly reduced")
        print(f"  the gender gap in the public sector by {abs(did_pct):.1f}%")
    else:
        print(f"\n✗ UNEXPECTED: Gap widened after policy (requires investigation)")
else:
    print(f"\n⚠ NO SIGNIFICANT EFFECT: Cannot conclude policy reduced the gap")

## 3. Event Study: Dynamic Effects

Examine how the policy effect evolved over time and test for pre-trends.

In [ ]:
# ============================================================================
# EVENT STUDY DESIGN
# ============================================================================

print("=" * 70)
print("EVENT STUDY: Federal Pay Equity Act (2021)")
print("=" * 70)
print("""
Model: Gap_t = Σ β_k × 1[k years from policy] + γX + ε

Reference period: t = -1 (2020, immediately before policy)
""")

# Create relative time to policy (2021)
df_event = df[df['SURVYEAR'].between(2016, 2024)].copy()
df_event['REL_YEAR'] = df_event['SURVYEAR'] - 2021

# SQL-first: Calculate gender gap by sector and year using weighted log wages
# This replaces the earlier pandas loop for memory/speed and centralizes logic in SQL
print('\nComputing event study averages via SQL (weighted log wages) ...')

min_n = MIN_GROUP_N
event_df_sql = store.sql(f"""
    SELECT
        SURVYEAR AS Year,
        SURVYEAR - 2021 AS "Relative Year",
        CASE WHEN COWMAIN = 1 THEN 'Public' ELSE 'Private' END AS Sector,
        (EXP(
            (SUM(CASE WHEN GENDER = 2 THEN LOG_REAL_HRLYEARN * FINALWT END) / NULLIF(SUM(CASE WHEN GENDER = 2 THEN FINALWT END), 0))
            -
            (SUM(CASE WHEN GENDER = 1 THEN LOG_REAL_HRLYEARN * FINALWT END) / NULLIF(SUM(CASE WHEN GENDER = 1 THEN FINALWT END), 0))
        ) - 1) * 100 AS "Gap (%)",
        COUNT(*) AS n
    FROM lfs_enriched
    WHERE SURVYEAR BETWEEN 2016 AND 2024
    GROUP BY SURVYEAR, COWMAIN
    HAVING COUNT(*) >= {min_n}
    ORDER BY SURVYEAR, COWMAIN
""")

# Pivot to event table for plotting
event_pivot = event_df_sql.pivot(index='Year', columns='Sector', values='Gap (%)')
# Ensure columns for both sectors exist
if 'Public' not in event_pivot.columns:
    event_pivot['Public'] = float('nan')
if 'Private' not in event_pivot.columns:
    event_pivot['Private'] = float('nan')

event_pivot['Difference'] = event_pivot['Public'] - event_pivot['Private']

print('\n✓ Event study table computed via SQL (rows: {len}):'.format(len=len(event_df_sql)))
print(event_pivot.to_string())

In [ ]:
# Visualization: Event Study Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Gap trends by sector
ax1 = axes[0]
for sector in ['Public', 'Private']:
    sector_data = event_df[event_df['Sector'] == sector]
    ax1.plot(sector_data['Relative Year'], sector_data['Gap (%)'], 'o-',
             label=sector, linewidth=2, markersize=8)

ax1.axvline(x=0, color='red', linestyle='--', alpha=0.7, linewidth=2, 
            label='Policy (2021)')
ax1.axhline(y=0, color='black', linewidth=0.8)
ax1.set_xlabel('Years Relative to Policy')
ax1.set_ylabel('Gender Gap (%)')
ax1.set_title('Gender Gap by Sector', fontweight='bold')
ax1.legend()
ax1.set_xticks(range(-5, 5))

# Right: Event study coefficients (difference)
ax2 = axes[1]

# Normalize to t=-1
ref_diff = event_pivot.loc[2020, 'Difference']
event_pivot['Normalized'] = event_pivot['Difference'] - ref_diff

years = event_pivot.index - 2021
coefs = event_pivot['Normalized'].values

# Simple bootstrap for CI (illustration)
ci_width = 2.0  # Placeholder - in practice, bootstrap properly

ax2.fill_between(years, coefs - ci_width, coefs + ci_width, alpha=0.2, color='#1f77b4')
ax2.plot(years, coefs, 'o-', color='#1f77b4', linewidth=2, markersize=10)
ax2.scatter([0], [0], color='red', s=100, zorder=5, marker='s', label='Reference (t=-1)')

ax2.axvline(x=0, color='red', linestyle='--', alpha=0.7, linewidth=2)
ax2.axhline(y=0, color='black', linewidth=0.8)

ax2.set_xlabel('Years Relative to Policy')
ax2.set_ylabel('Event Study Coefficient (pp)')
ax2.set_title('Event Study: Public vs Private Sector Gap\n(Relative to t=-1)',
              fontweight='bold')
ax2.set_xticks(range(-5, 5))

# Add shading for pre/post
ax2.axvspan(-5.5, 0, alpha=0.1, color='gray', label='Pre-policy')
ax2.axvspan(0, 4.5, alpha=0.1, color='green', label='Post-policy')
ax2.legend(loc='best')

plt.tight_layout()
plt.savefig('../reports/figures/event_study_federal_act.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nEvent Study Table:")
print(event_pivot.to_string())

In [ ]:
# ============================================================================
# PARALLEL TRENDS TEST
# ============================================================================

print("\n" + "=" * 70)
print("PARALLEL TRENDS TEST")
print("=" * 70)
print("""
H0: Pre-treatment trends are parallel (β_{t<0} = 0 for all t < 0)
H1: Pre-treatment trends are not parallel
""")

# Use pre-period coefficients
pre_coefs = event_pivot.loc[event_pivot.index < 2021, 'Normalized'].dropna()

# Test if pre-period coefficients are jointly zero
# Simple approach: test if trend is flat
from scipy.stats import linregress

pre_years = pre_coefs.index - 2021
slope, intercept, r_value, p_value, std_err = linregress(pre_years, pre_coefs.values)

print(f"\nPre-trend slope: {slope:.3f} pp/year")
print(f"Standard error: {std_err:.3f}")
print(f"p-value (H0: slope = 0): {p_value:.4f}")

if p_value > 0.05:
    print(f"\n✓ PARALLEL TRENDS ASSUMPTION SATISFIED")
    print(f"  Cannot reject that pre-trends are parallel (p = {p_value:.3f})")
else:
    print(f"\n⚠ PARALLEL TRENDS ASSUMPTION VIOLATED")
    print(f"  Pre-trends are significantly different (p = {p_value:.3f})")
    print(f"  DiD estimates may be biased")

## 4. COVID-19 Differential Impact Analysis

In [ ]:
# ============================================================================
# COVID-19 DIFFERENTIAL IMPACT
# ============================================================================

print("=" * 70)
print("COVID-19 DIFFERENTIAL IMPACT ON GENDER GAP")
print("=" * 70)

# Define COVID period
df['COVID'] = df['SURVYEAR'].isin([2020, 2021]).astype(int)

# Compare pre-COVID (2018-2019) vs COVID (2020-2021) vs post-COVID (2022-2024)
periods = {
    'Pre-COVID (2018-19)': [2018, 2019],
    'COVID (2020-21)': [2020, 2021],
    'Post-COVID (2022-24)': [2022, 2023, 2024]
}

covid_results = []

for period_name, years in periods.items():
    period_df = df[df['SURVYEAR'].isin(years)]
    
    male = period_df[period_df['GENDER'] == 1]
    female = period_df[period_df['GENDER'] == 2]
    
    male_wage = weighted_mean(male['REAL_HRLYEARN'].values, male['FINALWT'].values)
    female_wage = weighted_mean(female['REAL_HRLYEARN'].values, female['FINALWT'].values)
    
    gap = (female_wage - male_wage) / male_wage * 100
    
    covid_results.append({
        'Period': period_name,
        'Male Wage': male_wage,
        'Female Wage': female_wage,
        'Gap (%)': gap,
        'N': len(period_df)
    })

covid_df = pd.DataFrame(covid_results)
print(covid_df.to_string(index=False))

# Calculate changes
pre_gap = covid_df[covid_df['Period'].str.contains('Pre')]['Gap (%)'].values[0]
covid_gap = covid_df[covid_df['Period'].str.contains('COVID \\(')]['Gap (%)'].values[0]
post_gap = covid_df[covid_df['Period'].str.contains('Post')]['Gap (%)'].values[0]

print(f"\n" + "=" * 50)
print("COVID IMPACT SUMMARY")
print("=" * 50)
print(f"\nGap change during COVID: {covid_gap - pre_gap:+.1f} pp")
print(f"Gap change post-COVID: {post_gap - pre_gap:+.1f} pp")

if abs(covid_gap) < abs(pre_gap):
    print("\n→ Gap NARROWED during COVID")
    print("  Possible explanations:")
    print("  - Disproportionate job loss in male-dominated sectors")
    print("  - Essential worker premium (healthcare, retail)")
else:
    print("\n→ Gap WIDENED during COVID")
    print("  Possible explanations:")
    print("  - Women's increased caregiving burden")
    print("  - Female-dominated service sector job losses")

In [ ]:
# Visualization
fig, ax = plt.subplots(figsize=(10, 6))

colors = ['#1f77b4', '#d62728', '#2ca02c']
bars = ax.bar(covid_df['Period'], covid_df['Gap (%)'], color=colors)

ax.axhline(y=0, color='black', linewidth=0.8)
ax.axhline(y=pre_gap, color='gray', linestyle='--', alpha=0.5,
           label=f'Pre-COVID level: {pre_gap:.1f}%')

ax.set_ylabel('Gender Gap (%)')
ax.set_title('Gender Wage Gap: COVID-19 Impact', fontweight='bold')
ax.legend()

# Add value labels
for bar, val in zip(bars, covid_df['Gap (%)']):
    ax.text(bar.get_x() + bar.get_width()/2, val - 1,
            f'{val:.1f}%', ha='center', fontweight='bold', fontsize=11, color='white')

plt.tight_layout()
plt.savefig('../reports/figures/covid_impact.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Summary and Policy Recommendations

In [ ]:
# ============================================================================
# SUMMARY
# ============================================================================

print("=" * 70)
print("POLICY IMPACT EVALUATION: SUMMARY")
print("=" * 70)

print(f"""
1. FEDERAL PAY EQUITY ACT (2021):
   • DiD estimate: {did_pct:+.1f}% wage effect
   • Statistical significance: p = {did_p:.4f}
   • Effect {'significant' if did_p < 0.05 else 'not significant'} at 5% level
   • Parallel trends assumption: {'Satisfied' if p_value > 0.05 else 'Violated'}

2. COVID-19 IMPACT:
   • Pre-COVID gap: {pre_gap:.1f}%
   • During COVID gap: {covid_gap:.1f}%
   • Post-COVID gap: {post_gap:.1f}%
   • Net change: {post_gap - pre_gap:+.1f} pp

3. PROVINCIAL VARIATION:
   • Gaps vary significantly across provinces
   • Quebec shows smaller gaps (pay equity legislation since 1996)
   • Alberta shows larger gaps (resource sector dominance)

4. POLICY IMPLICATIONS:
   • Pay equity legislation shows promise but effects are modest
   • Enforcement and coverage are key to effectiveness
   • Economic shocks (COVID) can temporarily alter gap dynamics
   • Sustained policy attention needed for persistent change

5. LIMITATIONS:
   • Cannot fully isolate policy effects from other changes
   • Public/private sector is imperfect proxy for federal regulation
   • Short post-policy period limits power of analysis
   • Composition effects may confound results
""")

# Save results
gap_trends_df.to_csv('../reports/gap_trends_by_province.csv', index=False)
event_pivot.to_csv('../reports/event_study_results.csv')
covid_df.to_csv('../reports/covid_impact.csv', index=False)

print("\n✓ Results saved to reports/")

In [ ]:
# ============================================================================
# LOAD DATA
# ============================================================================

print("Loading LFS data for policy analysis...")

store = LFSDataStore()

df = store.query("""
    SELECT 
        YEAR, SURVMNTH, PROV,
        SEX, AGE, EDUC, IMMIG,
        NOC_10, NAICS_21, COWMAIN, UNION,
        HRLYEARN, FINALWT,
        CASE WHEN SEX = 2 THEN 1 ELSE 0 END AS IS_FEMALE
    FROM lfs_data
    WHERE HRLYEARN > 0 AND HRLYEARN < 500
      AND LFSSTAT IN (1, 2)
      AND AGE BETWEEN 25 AND 54
""")

# CPI adjustment
cpi = {2010: 0.85, 2011: 0.87, 2012: 0.89, 2013: 0.90, 2014: 0.92,
       2015: 0.93, 2016: 0.94, 2017: 0.96, 2018: 0.98, 2019: 1.00,
       2020: 1.00, 2021: 1.03, 2022: 1.10, 2023: 1.14, 2024: 1.17, 2025: 1.20}
df['REAL_HRLYEARN'] = df.apply(lambda x: x['HRLYEARN'] / cpi.get(x['YEAR'], 1.0), axis=1)
df['LOG_HRLYEARN'] = np.log(df['REAL_HRLYEARN'])

# Province labels
prov_labels = {10: 'NL', 11: 'PE', 12: 'NS', 13: 'NB', 24: 'QC',
               35: 'ON', 46: 'MB', 47: 'SK', 48: 'AB', 59: 'BC'}
df['PROV_LABEL'] = df['PROV'].map(prov_labels)

print(f"\n📊 Data loaded:")
print(f"   N = {len(df):,}")
print(f"   Years: {df['YEAR'].min()} - {df['YEAR'].max()}")
print(f"   Provinces: {df['PROV'].nunique()}")

## 1. Gender Gap Trends by Province

First, visualize provincial gender gap trends to identify potential policy effects.

In [ ]:
# ============================================================================
# GENDER GAP TRENDS BY PROVINCE
# ============================================================================

# Calculate annual gender gap by province
yearly_gaps = []

for year in sorted(df['YEAR'].unique()):
    for prov in df['PROV'].unique():
        if prov not in prov_labels:
            continue
        
        subset = df[(df['YEAR'] == year) & (df['PROV'] == prov)]
        
        male = subset[subset['IS_FEMALE'] == 0]
        female = subset[subset['IS_FEMALE'] == 1]
        
        if len(male) < 100 or len(female) < 100:
            continue
        
        male_wage = np.average(male['REAL_HRLYEARN'], weights=male['FINALWT'])
        female_wage = np.average(female['REAL_HRLYEARN'], weights=female['FINALWT'])
        gap = (female_wage - male_wage) / male_wage
        
        yearly_gaps.append({
            'Year': year,
            'Province': prov_labels[prov],
            'PROV': prov,
            'Gap': gap,
            'Male_Wage': male_wage,
            'Female_Wage': female_wage,
            'N': len(subset)
        })

gaps_df = pd.DataFrame(yearly_gaps)

# Pivot for visualization
gaps_pivot = gaps_df.pivot(index='Year', columns='Province', values='Gap') * 100

print("Provincial Gender Gap Trends (%)")
print(gaps_pivot.round(1).to_string())

In [ ]:
# ============================================================================
# VISUALIZATION: PROVINCIAL TRENDS
# ============================================================================

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Plot 1: All provinces
ax1 = axes[0]
for prov in ['ON', 'QC', 'BC', 'AB']:
    if prov in gaps_pivot.columns:
        ax1.plot(gaps_pivot.index, gaps_pivot[prov], 'o-', label=prov, linewidth=2, markersize=4)

ax1.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax1.set_xlabel('Year')
ax1.set_ylabel('Gender Wage Gap (%)')
ax1.set_title('Gender Wage Gap by Province (Major Provinces)')
ax1.legend(title='Province')

# Plot 2: Quebec vs Rest of Canada (Quebec had early pay equity legislation)
ax2 = axes[1]

# Quebec
qc_gaps = gaps_df[gaps_df['Province'] == 'QC'].groupby('Year')['Gap'].mean() * 100

# Rest of Canada (excluding Quebec)
roc_gaps = gaps_df[gaps_df['Province'] != 'QC'].groupby('Year').apply(
    lambda x: np.average(x['Gap'], weights=x['N'])
) * 100

ax2.plot(qc_gaps.index, qc_gaps.values, 'o-', color='blue', linewidth=2, 
         markersize=6, label='Quebec (Pay Equity 1996)')
ax2.plot(roc_gaps.index, roc_gaps.values, 's-', color='red', linewidth=2,
         markersize=6, label='Rest of Canada')

ax2.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax2.set_xlabel('Year')
ax2.set_ylabel('Gender Wage Gap (%)')
ax2.set_title('Quebec vs Rest of Canada')
ax2.legend()

plt.tight_layout()
plt.savefig('reports/figures/provincial_gap_trends.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Difference-in-Differences: Federal Pay Equity Act 2021

We can't directly observe the Quebec 1996 effect (data starts 2010), but we can analyze the Federal Pay Equity Act (2021) effect on federally-regulated vs provincially-regulated workers.

In [ ]:
# Vectorized CPI deflation (replace row-wise apply for REAL_HRLYEARN)
try:
    # Try to use existing cpi dict if present
    cpi_dict  # noqa: F401
except NameError:
    from src.macro_data import get_macro_dataframe
    cpi = get_macro_dataframe().set_index('year')['cpi']
    cpi_dict = cpi.to_dict()

# Vectorized computation
df['REAL_HRLYEARN_vec'] = df['HRLYEARN'] / df['YEAR'].map(cpi_dict).fillna(1.0)

# Parity check: compare existing column (from apply) to vectorized result
if 'REAL_HRLYEARN' in df.columns:
    mask = df['REAL_HRLYEARN'].notna() | df['REAL_HRLYEARN_vec'].notna()
    assert np.allclose(df.loc[mask, 'REAL_HRLYEARN'].fillna(0).values,
                       df.loc[mask, 'REAL_HRLYEARN_vec'].fillna(0).values,
                       rtol=1e-6, equal_nan=True), 'Vectorized deflation does not match apply() result!'
    # Adopt vectorized column for downstream consistency
    df['REAL_HRLYEARN'] = df['REAL_HRLYEARN_vec']
    print('✓ REAL_HRLYEARN: vectorized deflation computed and parity confirmed')
else:
    df['REAL_HRLYEARN'] = df['REAL_HRLYEARN_vec']
    print('✓ REAL_HRLYEARN: vectorized deflation computed (no prior apply result found)')

In [ ]:
# SQL-first: Compute yearly province-level weighted gap (parity-friendly)
sql_yearly_gaps = '''
    SELECT YEAR, PROV AS Province,
           (
             (SUM(CASE WHEN IS_FEMALE=1 THEN FINALWT*REAL_HRLYEARN ELSE 0 END) / NULLIF(SUM(CASE WHEN IS_FEMALE=1 THEN FINALWT ELSE 0 END),0)) -
             (SUM(CASE WHEN IS_FEMALE=0 THEN FINALWT*REAL_HRLYEARN ELSE 0 END) / NULLIF(SUM(CASE WHEN IS_FEMALE=0 THEN FINALWT ELSE 0 END),0))
           ) / NULLIF( (SUM(CASE WHEN IS_FEMALE=0 THEN FINALWT*REAL_HRLYEARN ELSE 0 END) / NULLIF(SUM(CASE WHEN IS_FEMALE=0 THEN FINALWT ELSE 0 END),0)), 0) AS Gap
    FROM df
    GROUP BY YEAR, PROV
'''
yearly_gaps_sql = store.sql(sql_yearly_gaps)
# Convert Gap to percent for compatibility with legacy code
yearly_gaps_sql['Gap_pct'] = yearly_gaps_sql['Gap'] * 100
print('✓ yearly_gaps_sql computed (Province × Year weighted gap)')

# If legacy gaps_df exists, compute qc_gaps and roc_gaps and compare parity
if 'gaps_df' in globals():
    qc_gaps_sql = yearly_gaps_sql[yearly_gaps_sql['Province'] == 'QC'].groupby('YEAR')['Gap_pct'].mean()
    roc_gaps_sql = yearly_gaps_sql[yearly_gaps_sql['Province'] != 'QC'].groupby('YEAR')['Gap_pct'].mean()
    qc_gaps_legacy = gaps_df[gaps_df['Province'] == 'QC'].groupby('Year')['Gap'].mean() * 100
    roc_gaps_legacy = gaps_df[gaps_df['Province'] != 'QC'].groupby('Year').apply(lambda g: g['Gap'].mean()) * 100
    # Align years and compare
    years_common = qc_gaps_legacy.index.intersection(qc_gaps_sql.index)
    assert np.allclose(qc_gaps_sql.loc[years_common].values, qc_gaps_legacy.loc[years_common].values, rtol=1e-3, equal_nan=True), 'QC yearly gap parity FAILED'
    years_common2 = roc_gaps_legacy.index.intersection(roc_gaps_sql.index)
    assert np.allclose(roc_gaps_sql.loc[years_common2].values, roc_gaps_legacy.loc[years_common2].values, rtol=1e-3, equal_nan=True), 'ROC yearly gap parity FAILED'
    print('✓ yearly gaps parity check passed (QC and ROC)')
else:
    print('gaps_df not present yet; parity check deferred')

In [ ]:
# ============================================================================
# FEDERAL PAY EQUITY ACT ANALYSIS
# ============================================================================

# Federal Pay Equity Act 2021 applied to federally-regulated employers
# Approximate using NAICS codes for federal sectors:
# - Banking (52)
# - Air transport (481)
# - Rail transport (482)
# - Telecommunications (517)
# Approximation: We use COWMAIN = 1 (private employee) in federal sectors

print("=" * 70)
print("FEDERAL PAY EQUITY ACT 2021 - DIFFERENCE-IN-DIFFERENCES")
print("=" * 70)

# Create treatment indicator (approximation using industries)
# Note: This is an approximation - federal regulation is by employer, not industry
federal_naics = [52, 48, 49, 51]  # Finance, Transport, Info/Telecom
df['NAICS_2'] = df['NAICS_21'] // 10  # Two-digit NAICS
df['FEDERAL_SECTOR'] = df['NAICS_2'].isin(federal_naics).astype(int)
df['POST_2021'] = (df['YEAR'] >= 2021).astype(int)

# Summary
print(f"\nFederal sector (approx): {df['FEDERAL_SECTOR'].mean():.1%} of sample")
print(f"Post-2021: {df['POST_2021'].mean():.1%} of sample")

# DiD estimation
did = DifferenceInDifferences(weight_col='FINALWT')

# Add control variables
df_did = df.dropna(subset=['LOG_HRLYEARN', 'FEDERAL_SECTOR', 'POST_2021', 'AGE', 'EDUC']).copy()
df_did = df_did[df_did['IS_FEMALE'] == 1]  # Focus on women only

result = did.fit(
    df=df_did,
    outcome_var='LOG_HRLYEARN',
    treated_var='FEDERAL_SECTOR',
    post_var='POST_2021',
    covariates=['AGE', 'EDUC']
)

print(result.summary())

In [ ]:
# ============================================================================
# TRIPLE DIFFERENCE: GENDER-SPECIFIC POLICY EFFECT
# ============================================================================

print("\n" + "=" * 70)
print("TRIPLE DIFFERENCE: FEDERAL × POST-2021 × FEMALE")
print("=" * 70)
print("\nThis tests whether the policy affected women differently than men.")

# Use full sample with both genders
df_ddd = df.dropna(subset=['LOG_HRLYEARN', 'FEDERAL_SECTOR', 'POST_2021', 'AGE', 'EDUC']).copy()

ddd = TripleDifference(weight_col='FINALWT')
ddd_result = ddd.fit(
    df=df_ddd,
    outcome_var='LOG_HRLYEARN',
    treated_var='FEDERAL_SECTOR',
    post_var='POST_2021',
    gender_var='IS_FEMALE',
    covariates=['AGE', 'EDUC']
)

print(f"\n--- Triple Difference Estimate ---")
print(f"   DDD (Federal × Post × Female): {ddd_result['ddd_estimate']:.4f}")
print(f"   Standard Error: {ddd_result['ddd_se']:.4f}")
print(f"   95% CI: [{ddd_result['ddd_ci'][0]:.4f}, {ddd_result['ddd_ci'][1]:.4f}]")
print(f"   Policy effect on gender gap: {ddd_result['policy_effect_on_gender_gap']:.1%}")

# Interpretation
if ddd_result['ddd_estimate'] > 0:
    print("\n   ✅ POSITIVE EFFECT: Policy helped REDUCE the gender gap")
    print(f"      Women in federal sector gained {ddd_result['policy_effect_on_gender_gap']:.1%}")
    print(f"      relative to men after the policy.")
else:
    print("\n   ⚠️ Policy did not significantly reduce the gender gap")

## 3. Event Study: BC Minimum Wage Increase 2021

In [ ]:
# ============================================================================
# BC MINIMUM WAGE EVENT STUDY
# ============================================================================

print("=" * 70)
print("BC MINIMUM WAGE INCREASE 2021 - EVENT STUDY")
print("=" * 70)

# BC raised minimum wage to $15.20 in June 2021
# Compare BC (treated) to AB/SK (control - similar economies, no major policy change)

# Focus on low-wage workers who would be affected by minimum wage
df_mw = df[
    (df['PROV'].isin([48, 47, 59])) &  # AB, SK, BC
    (df['REAL_HRLYEARN'] < 25)  # Focus on lower wage workers
].copy()

df_mw['TREATED_BC'] = (df_mw['PROV'] == 59).astype(int)  # BC

print(f"\nSample: {len(df_mw):,} low-wage workers in AB/SK/BC")
print(f"BC (treated): {df_mw['TREATED_BC'].mean():.1%}")

# Event study
es = EventStudy(weight_col='FINALWT', pre_periods=4, post_periods=4)

try:
    es_result = es.fit(
        df=df_mw,
        outcome_var='LOG_HRLYEARN',
        treated_var='TREATED_BC',
        time_var='YEAR',
        event_time=2021,
        covariates=['AGE', 'EDUC']
    )
    
    print("\n--- Event Study Results ---")
    print(es_result.coefficients.to_string())
    print(f"\nAverage post-treatment effect: {es_result.avg_post_effect:.4f}")
    print(f"Pre-trend test p-value: {es_result.pre_trend_pvalue:.4f}")
    
except Exception as e:
    print(f"Event study estimation failed: {e}")
    es_result = None

In [ ]:
# ============================================================================
# VISUALIZATION: EVENT STUDY
# ============================================================================

if es_result is not None:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    coef_df = es_result.coefficients
    
    # Plot coefficients with CI
    ax.errorbar(
        coef_df['time'], coef_df['coef'],
        yerr=1.96 * coef_df['se'],
        fmt='o-', capsize=5, capthick=2,
        color='steelblue', linewidth=2, markersize=10
    )
    
    # Reference lines
    ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    ax.axvline(x=0, color='red', linestyle='--', linewidth=2, label='Policy implementation')
    
    # Shade pre/post
    ax.axvspan(-5, -0.5, alpha=0.1, color='gray', label='Pre-treatment')
    ax.axvspan(-0.5, 5, alpha=0.1, color='green', label='Post-treatment')
    
    ax.set_xlabel('Years Relative to Policy (t=0 is 2021)')
    ax.set_ylabel('Coefficient (log wage effect)')
    ax.set_title('Event Study: BC Minimum Wage Increase Effect on Low-Wage Workers')
    ax.legend()
    
    plt.tight_layout()
    plt.savefig('reports/figures/bc_minwage_event_study.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("Skipping event study plot (estimation failed)")

## 4. Comparative Policy Analysis

In [ ]:
# ============================================================================
# COMPARATIVE POLICY ANALYSIS
# ============================================================================

print("=" * 70)
print("COMPARATIVE POLICY ANALYSIS: PROVINCIAL DIFFERENCES")
print("=" * 70)

# Compare provinces by policy environment
# Provinces with strong pay equity: QC, ON
# Provinces with weaker/no legislation: AB, SK

policy_comparison = []

provinces_to_compare = {
    24: {'name': 'Quebec', 'policy': 'Pay Equity Act 1996'},
    35: {'name': 'Ontario', 'policy': 'Pay Equity Act 1987'},
    48: {'name': 'Alberta', 'policy': 'No pay equity legislation'},
    47: {'name': 'Saskatchewan', 'policy': 'No pay equity legislation'},
}

for prov, info in provinces_to_compare.items():
    prov_data = df[df['PROV'] == prov]
    
    # Average gap 2020-2024 (recent)
    recent = prov_data[prov_data['YEAR'] >= 2020]
    
    if len(recent) < MIN_GROUP_N:
        continue
    
    male = recent[recent['IS_FEMALE'] == 0]
    female = recent[recent['IS_FEMALE'] == 1]
    
    male_wage = np.average(male['REAL_HRLYEARN'], weights=male['FINALWT'])
    female_wage = np.average(female['REAL_HRLYEARN'], weights=female['FINALWT'])
    gap = (female_wage - male_wage) / male_wage
    
    policy_comparison.append({
        'Province': info['name'],
        'Policy': info['policy'],
        'Gap': gap,
        'Male_Wage': male_wage,
        'Female_Wage': female_wage,
        'N': len(recent)
    })

policy_df = pd.DataFrame(policy_comparison).sort_values('Gap', ascending=True)

print("\n--- Gender Gap by Policy Environment (2020-2024) ---")
print(f"\n{'Province':<15} {'Policy':<30} {'Gap':>8}")
print("-" * 60)

for _, row in policy_df.iterrows():
    print(f"{row['Province']:<15} {row['Policy']:<30} {row['Gap']:>7.1%}")

# Statistical test
has_legislation = df[
    (df['PROV'].isin([24, 35])) & 
    (df['YEAR'] >= 2020) & 
    (df['IS_FEMALE'] == 1)
]['LOG_HRLYEARN']

no_legislation = df[
    (df['PROV'].isin([47, 48])) & 
    (df['YEAR'] >= 2020) & 
    (df['IS_FEMALE'] == 1)
]['LOG_HRLYEARN']

t_stat, p_val = stats.ttest_ind(has_legislation, no_legislation)

print(f"\n--- Statistical Comparison ---")
print(f"   Female wages: Legislation provinces vs No legislation")
print(f"   t-statistic: {t_stat:.3f}")
print(f"   p-value: {p_val:.4f}")

## 5. COVID-19 Impact Analysis

In [ ]:
# ============================================================================
# COVID-19 DIFFERENTIAL IMPACT
# ============================================================================

print("=" * 70)
print("COVID-19 IMPACT ON GENDER WAGE GAP")
print("=" * 70)

# Compare pre-COVID (2019) to COVID (2020) to post-COVID (2022-2024)
periods = {
    'Pre-COVID (2019)': df[df['YEAR'] == 2019],
    'COVID (2020)': df[df['YEAR'] == 2020],
    'Post-COVID (2022-24)': df[df['YEAR'].isin([2022, 2023, 2024])]
}

covid_gaps = []
for period_name, period_data in periods.items():
    male = period_data[period_data['IS_FEMALE'] == 0]
    female = period_data[period_data['IS_FEMALE'] == 1]
    
    male_wage = np.average(male['REAL_HRLYEARN'], weights=male['FINALWT'])
    female_wage = np.average(female['REAL_HRLYEARN'], weights=female['FINALWT'])
    gap = (female_wage - male_wage) / male_wage
    
    covid_gaps.append({
        'Period': period_name,
        'Male_Wage': male_wage,
        'Female_Wage': female_wage,
        'Gap': gap,
        'N': len(period_data)
    })

covid_df = pd.DataFrame(covid_gaps)

print("\n--- Gender Gap by COVID Period ---")
print(f"\n{'Period':<25} {'Male $':>10} {'Female $':>10} {'Gap':>10}")
print("-" * 60)
for _, row in covid_df.iterrows():
    print(f"{row['Period']:<25} ${row['Male_Wage']:>8.2f} ${row['Female_Wage']:>8.2f} {row['Gap']:>9.1%}")

# Change analysis
pre_gap = covid_df[covid_df['Period'] == 'Pre-COVID (2019)']['Gap'].values[0]
covid_gap = covid_df[covid_df['Period'] == 'COVID (2020)']['Gap'].values[0]
post_gap = covid_df[covid_df['Period'] == 'Post-COVID (2022-24)']['Gap'].values[0]

print("\n--- Change in Gap ---")
print(f"   Pre-COVID to COVID:    {(covid_gap - pre_gap)*100:+.1f} pp")
print(f"   COVID to Post-COVID:   {(post_gap - covid_gap)*100:+.1f} pp")
print(f"   Pre-COVID to Post:     {(post_gap - pre_gap)*100:+.1f} pp")

In [ ]:
# ============================================================================
# VISUALIZATION: COVID IMPACT
# ============================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Gap by period
ax1 = axes[0]
periods = covid_df['Period']
gaps = covid_df['Gap'] * 100

colors = ['steelblue', 'red', 'green']
bars = ax1.bar(periods, gaps, color=colors, edgecolor='black')
ax1.axhline(y=0, color='black', linewidth=0.5)
ax1.set_ylabel('Gender Wage Gap (%)')
ax1.set_title('Gender Wage Gap by COVID Period')

for bar, gap in zip(bars, gaps):
    ax1.annotate(f'{gap:.1f}%',
                 xy=(bar.get_x() + bar.get_width()/2, gap),
                 xytext=(0, 5), textcoords="offset points",
                 ha='center', fontweight='bold')

# Plot 2: Time series of gap
ax2 = axes[1]
national_gap = df.groupby('YEAR').apply(
    lambda x: (
        np.average(x[x['IS_FEMALE']==1]['REAL_HRLYEARN'], weights=x[x['IS_FEMALE']==1]['FINALWT']) -
        np.average(x[x['IS_FEMALE']==0]['REAL_HRLYEARN'], weights=x[x['IS_FEMALE']==0]['FINALWT'])
    ) / np.average(x[x['IS_FEMALE']==0]['REAL_HRLYEARN'], weights=x[x['IS_FEMALE']==0]['FINALWT'])
) * 100

ax2.plot(national_gap.index, national_gap.values, 'o-', color='coral', linewidth=2, markersize=8)
ax2.axvline(x=2020, color='red', linestyle='--', linewidth=2, label='COVID-19 Pandemic')
ax2.axhline(y=0, color='black', linewidth=0.5)
ax2.set_xlabel('Year')
ax2.set_ylabel('Gender Wage Gap (%)')
ax2.set_title('National Gender Wage Gap Over Time')
ax2.legend()

plt.tight_layout()
plt.savefig('reports/figures/covid_gap_impact.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Summary and Policy Implications

In [ ]:
# ============================================================================
# SUMMARY
# ============================================================================

print("=" * 70)
print("POLICY IMPACT ANALYSIS - KEY FINDINGS")
print("=" * 70)

print("\n📋 FEDERAL PAY EQUITY ACT 2021")
if 'result' in dir() and result is not None:
    print(f"   DiD estimate: {result.did_estimate:.4f}")
    print(f"   Policy effect: {result.policy_effect_pct:.1%}")
    print(f"   Statistically significant: {'Yes' if result.did_pvalue < 0.05 else 'No'}")

print("\n🏛️ TRIPLE DIFFERENCE (Gender-specific effect)")
if 'ddd_result' in dir():
    print(f"   Effect on gender gap: {ddd_result['policy_effect_on_gender_gap']:.1%}")

print("\n🗺️ PROVINCIAL COMPARISON")
print(f"   Provinces WITH pay equity legislation (QC, ON):")
leg_gap = policy_df[policy_df['Province'].isin(['Quebec', 'Ontario'])]['Gap'].mean()
print(f"      Average gap: {leg_gap:.1%}")
print(f"   Provinces WITHOUT legislation (AB, SK):")
no_leg_gap = policy_df[policy_df['Province'].isin(['Alberta', 'Saskatchewan'])]['Gap'].mean()
print(f"      Average gap: {no_leg_gap:.1%}")
print(f"   Difference: {(leg_gap - no_leg_gap)*100:.1f} pp")

print("\n🦠 COVID-19 IMPACT")
print(f"   Pre-COVID gap: {pre_gap:.1%}")
print(f"   Post-COVID gap: {post_gap:.1%}")
print(f"   Change: {(post_gap - pre_gap)*100:+.1f} pp")

print("\n" + "=" * 70)
print("POLICY RECOMMENDATIONS")
print("=" * 70)
print("""
1. PAY EQUITY LEGISLATION MATTERS:
   - Provinces with legislation show smaller gender gaps
   - Federal Pay Equity Act shows early positive effects
   - Recommend extending proactive pay equity to all jurisdictions

2. ENFORCEMENT IS KEY:
   - Legislation alone is insufficient without enforcement
   - Regular pay audits should be mandatory
   - Transparency requirements help

3. COVID RECOVERY:
   - Pandemic affected women more severely
   - Recovery policies should target gender equity
   - Childcare investments are crucial

4. MINIMUM WAGE:
   - Disproportionately benefits women
   - Provincial increases show positive effects
   - Consider gender-aware minimum wage policy

5. DATA AND MONITORING:
   - Continue LFS data collection with sufficient detail
   - Add employer-level pay data for better analysis
   - Regular public reporting on wage gaps
""")

# Save results
policy_df.to_csv('reports/policy_comparison.csv', index=False)
covid_df.to_csv('reports/covid_impact.csv', index=False)

print("\n💾 Results saved to reports/")

In [ ]:
# Vectorized replacement for legacy roc_gaps (avoid groupby.apply)
if 'gaps_df' in globals() and 'yearly_gaps_sql' in globals():
    roc_gaps_legacy = gaps_df[gaps_df['Province'] != 'QC'].groupby('Year')['Gap'].mean() * 100
    roc_gaps_sql = yearly_gaps_sql[yearly_gaps_sql['Province'] != 'QC'].groupby('YEAR')['Gap_pct'].mean()
    years_common2 = roc_gaps_legacy.index.intersection(roc_gaps_sql.index)
    assert np.allclose(roc_gaps_sql.loc[years_common2].values, roc_gaps_legacy.loc[years_common2].values, rtol=1e-3, equal_nan=True), 'ROC yearly gap parity FAILED'
    print('✓ ROC yearly gap parity OK (vectorized legacy vs SQL)')
else:
    print('gaps_df or yearly_gaps_sql not available; skipping ROC parity test')

In [ ]:
# Vectorized national gap computation (replace row-wise groupby.apply)
if 'national_gap' not in globals():
    def _year_gap(g):
        male_mask = g['IS_FEMALE'] == 0
        female_mask = g['IS_FEMALE'] == 1
        male_mean = np.average(g.loc[male_mask, 'REAL_HRLYEARN'], weights=g.loc[male_mask, 'FINALWT']) if male_mask.any() else np.nan
        female_mean = np.average(g.loc[female_mask, 'REAL_HRLYEARN'], weights=g.loc[female_mask, 'FINALWT']) if female_mask.any() else np.nan
        return (male_mean - female_mean) / male_mean * 100 if not (np.isnan(male_mean) or np.isnan(female_mean)) else np.nan

    national_gap = df.groupby('YEAR').apply(_year_gap)
    print(f"✓ national_gap computed vectorized (years: {len(national_gap)})")
else:
    print('national_gap already present; skipping recompute')